In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
# Business purpose:
# Notebook 1 created the synthetic oncology market.
# Notebook 2 loads those outputs and converts them into a forecasting dataset.
#
# Main idea:
# At forecast origin tau, a model can only use information that was available then.

try:
    import duckdb
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "duckdb", "-q"])
    import duckdb

import numpy as np
import pandas as pd

FORECAST_HORIZONS = [1, 3, 6, 12]


required_files = [
    "assumption_registry.csv",
    "epidemiology_monthly.csv",
    "patient_segment_inputs.csv",
    "therapy_eligibility.csv",
    "market_calendar_events.csv",
    "market_access_monthly.csv",
    "therapy_overlap.csv",
    "patient_flow.csv",
    "commercial_demand.csv",
    "data_dictionary.csv",
    "variable_classification.csv",
    "forecast_information_set_policy.csv",
]

missing_files = [file for file in required_files if file not in uploaded.keys()]

assert len(missing_files) == 0, f"Missing uploaded files: {missing_files}"

assumption_registry = pd.read_csv("assumption_registry.csv")
epidemiology_monthly = pd.read_csv("epidemiology_monthly.csv", parse_dates=["month"])
patient_segment_inputs = pd.read_csv("patient_segment_inputs.csv", parse_dates=["month"])
therapy_eligibility = pd.read_csv("therapy_eligibility.csv")
market_calendar_events = pd.read_csv("market_calendar_events.csv")
market_access_monthly = pd.read_csv("market_access_monthly.csv", parse_dates=["month"])
therapy_overlap = pd.read_csv("therapy_overlap.csv")
patient_flow = pd.read_csv("patient_flow.csv", parse_dates=["month"])
commercial_demand = pd.read_csv("commercial_demand.csv", parse_dates=["month"])

data_dictionary = pd.read_csv("data_dictionary.csv")
variable_classification = pd.read_csv("variable_classification.csv")
forecast_information_set_policy = pd.read_csv("forecast_information_set_policy.csv")

print("Notebook 1 outputs loaded.")

input_table_shapes = pd.DataFrame([
    {"table": "assumption_registry", "shape": assumption_registry.shape},
    {"table": "epidemiology_monthly", "shape": epidemiology_monthly.shape},
    {"table": "patient_segment_inputs", "shape": patient_segment_inputs.shape},
    {"table": "therapy_eligibility", "shape": therapy_eligibility.shape},
    {"table": "market_calendar_events", "shape": market_calendar_events.shape},
    {"table": "market_access_monthly", "shape": market_access_monthly.shape},
    {"table": "therapy_overlap", "shape": therapy_overlap.shape},
    {"table": "patient_flow", "shape": patient_flow.shape},
    {"table": "commercial_demand", "shape": commercial_demand.shape},
])

display(input_table_shapes)

assert len(assumption_registry) == 31
assert len(epidemiology_monthly) == 576
assert len(patient_segment_inputs) == 2304
assert len(therapy_eligibility) == 16
assert len(market_calendar_events) == 6
assert len(market_access_monthly) == 2304
assert len(therapy_overlap) == 16
assert len(patient_flow) == 9216
assert len(commercial_demand) == 2304

con = duckdb.connect(database=":memory:")

con.register("assumption_registry", assumption_registry)
con.register("epidemiology_monthly", epidemiology_monthly)
con.register("patient_segment_inputs", patient_segment_inputs)
con.register("therapy_eligibility", therapy_eligibility)
con.register("market_calendar_events", market_calendar_events)
con.register("market_access_monthly", market_access_monthly)
con.register("therapy_overlap", therapy_overlap)
con.register("patient_flow", patient_flow)
con.register("commercial_demand", commercial_demand)

print("Notebook 1 input validation passed.")
print("DuckDB setup complete.")
print(f"Forecast horizons: {FORECAST_HORIZONS}")

In [ ]:
# Create one clean monthly table at the forecasting grain:
#
# therapy × region × month

# It is the source mart that combines commercial demand, access,# epidemiology, patient pool, and competitor pressure.

analytical_mart_sql = """
WITH eligible_patient_pool AS (
    SELECT
        psi.month,
        psi.month_index,
        psi.region,
        te.therapy,

        SUM(
            CASE
                WHEN te.eligible = 1
                THEN psi.new_first_line_diagnosis_inflow
                ELSE 0
            END
        ) AS eligible_new_1l_patient_pool,

        SUM(
            CASE
                WHEN te.eligible = 1
                THEN psi.initial_active_patients
                ELSE 0
            END
        ) AS eligible_initial_active_pool

    FROM patient_segment_inputs psi

    INNER JOIN therapy_eligibility te
        ON psi.biomarker_status = te.biomarker_status
       AND psi.line_of_therapy = te.line_of_therapy

    GROUP BY
        psi.month,
        psi.month_index,
        psi.region,
        te.therapy
),

competitor_pressure AS (
    SELECT
        ma_focal.month,
        ma_focal.month_index,
        ma_focal.region,
        ma_focal.therapy,

        SUM(
            CASE
                WHEN ma_comp.therapy_launched = TRUE
                 AND ma_comp.access_rate_truth > 0
                THEN ov.patient_segment_overlap * ma_comp.access_rate_truth
                ELSE 0
            END
        ) AS competitor_overlap_pressure_truth,

        MAX(
            CASE
                WHEN ma_comp.therapy_launched = TRUE
                 AND ma_comp.access_rate_truth > 0
                 AND ov.patient_segment_overlap > 0
                THEN ma_comp.access_known_from_month
                ELSE NULL
            END
        ) AS competitor_overlap_known_from_month

    FROM market_access_monthly ma_focal

    LEFT JOIN therapy_overlap ov
        ON ma_focal.therapy = ov.focal_therapy

    LEFT JOIN market_access_monthly ma_comp
        ON ma_focal.month = ma_comp.month
       AND ma_focal.region = ma_comp.region
       AND ov.competitor_therapy = ma_comp.therapy
       AND ma_focal.therapy <> ma_comp.therapy

    GROUP BY
        ma_focal.month,
        ma_focal.month_index,
        ma_focal.region,
        ma_focal.therapy
),

epidemiology_timing AS (
    SELECT
        month,
        month_index,
        region,
        population,
        new_diagnosed_nsclc_cases,
        new_metastatic_diagnoses,
        epidemiology_regime,

        CASE
            WHEN epidemiology_regime = 'baseline' THEN 1
            ELSE 109
        END AS epidemiology_known_from_month

    FROM epidemiology_monthly
)

SELECT
    cd.month,
    cd.month_index,
    cd.region,
    cd.therapy,

    cd.observed_sales_units,
    cd.latent_demand_units,
    cd.unmet_demand_units,
    cd.new_starts,
    cd.active_patients_end,
    cd.patient_months,
    cd.supply_constrained,
    cd.supply_fill_rate,
    cd.supply_known_from_month,
    cd.market_regime,

    ma.launch_month,
    ma.launch_known_from_month,
    ma.therapy_launched,
    ma.access_rate_truth,
    ma.access_regime,
    ma.access_known_from_month,

    ep.population,
    ep.new_diagnosed_nsclc_cases,
    ep.new_metastatic_diagnoses,
    ep.epidemiology_regime,
    ep.epidemiology_known_from_month,

    pool.eligible_new_1l_patient_pool,
    pool.eligible_initial_active_pool,

    cp.competitor_overlap_pressure_truth,
    cp.competitor_overlap_known_from_month

FROM commercial_demand cd

LEFT JOIN market_access_monthly ma
    ON cd.month = ma.month
   AND cd.month_index = ma.month_index
   AND cd.region = ma.region
   AND cd.therapy = ma.therapy

LEFT JOIN epidemiology_timing ep
    ON cd.month = ep.month
   AND cd.month_index = ep.month_index
   AND cd.region = ep.region

LEFT JOIN eligible_patient_pool pool
    ON cd.month = pool.month
   AND cd.month_index = pool.month_index
   AND cd.region = pool.region
   AND cd.therapy = pool.therapy

LEFT JOIN competitor_pressure cp
    ON cd.month = cp.month
   AND cd.month_index = cp.month_index
   AND cd.region = cp.region
   AND cd.therapy = cp.therapy

ORDER BY
    cd.therapy,
    cd.region,
    cd.month_index
"""

analytical_mart = con.execute(analytical_mart_sql).df()

print(f"Analytical mart created: {len(analytical_mart):,} rows")
display(analytical_mart.head())

con.register("analytical_mart", analytical_mart)

expected_rows = 144 * 4 * 4

assert len(analytical_mart) == expected_rows
assert analytical_mart.groupby(["therapy", "region", "month"]).size().max() == 1
assert analytical_mart["observed_sales_units"].notna().all()
assert analytical_mart["observed_sales_units"].ge(-1e-8).all()
assert analytical_mart["latent_demand_units"].ge(-1e-8).all()

print("Analytical mart validation checks passed.")

In [ ]:
# Business purpose:
# Turn the monthly mart into forecasting rows.
#
# Each row means:
#
# At forecast origin tau,
# predict observed sales at tau + h.
#
# We create historical features using only data up to tau.
# Then we create patient/market features only when known by tau.

modeling_dataset_sql = """
WITH historical_features AS (
    SELECT
        therapy,
        region,
        month AS forecast_origin_month,
        month_index AS forecast_origin_month_index,

        observed_sales_units AS sales_at_forecast_origin,

        LAG(observed_sales_units, 1) OVER (
            PARTITION BY therapy, region
            ORDER BY month_index
        ) AS sales_lag_1,

        LAG(observed_sales_units, 3) OVER (
            PARTITION BY therapy, region
            ORDER BY month_index
        ) AS sales_lag_3,

        LAG(observed_sales_units, 6) OVER (
            PARTITION BY therapy, region
            ORDER BY month_index
        ) AS sales_lag_6,

        LAG(observed_sales_units, 12) OVER (
            PARTITION BY therapy, region
            ORDER BY month_index
        ) AS sales_lag_12,

        AVG(observed_sales_units) OVER (
            PARTITION BY therapy, region
            ORDER BY month_index
            ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING
        ) AS rolling_mean_3,

        AVG(observed_sales_units) OVER (
            PARTITION BY therapy, region
            ORDER BY month_index
            ROWS BETWEEN 6 PRECEDING AND 1 PRECEDING
        ) AS rolling_mean_6,

        STDDEV_SAMP(observed_sales_units) OVER (
            PARTITION BY therapy, region
            ORDER BY month_index
            ROWS BETWEEN 6 PRECEDING AND 1 PRECEDING
        ) AS rolling_sd_6,

        observed_sales_units
        - LAG(observed_sales_units, 3) OVER (
            PARTITION BY therapy, region
            ORDER BY month_index
        ) AS sales_change_vs_3_months_ago,

        CASE
            WHEN therapy_launched = TRUE
            THEN month_index - launch_month + 1
            ELSE 0
        END AS therapy_age_months,

        EXTRACT(month FROM month) AS calendar_month,
        EXTRACT(year FROM month) AS calendar_year

    FROM analytical_mart
),

horizons AS (
    SELECT 1 AS horizon_months
    UNION ALL SELECT 3
    UNION ALL SELECT 6
    UNION ALL SELECT 12
),

origin_horizon_panel AS (
    SELECT
        hf.*,
        h.horizon_months,
        hf.forecast_origin_month_index + h.horizon_months AS target_month_index

    FROM historical_features hf
    CROSS JOIN horizons h
),

target_joined AS (
    SELECT
        ohp.*,

        target.month AS target_month,
        target.observed_sales_units AS target_observed_sales_units,
        target.latent_demand_units AS target_latent_demand_units,

        target.market_regime AS target_market_regime,
        target.supply_constrained AS target_supply_constrained,

        target.access_rate_truth AS target_access_rate_truth,
        target.access_known_from_month AS target_access_known_from_month,

        target.new_metastatic_diagnoses AS target_new_metastatic_diagnoses,
        target.epidemiology_known_from_month AS target_epidemiology_known_from_month,

        target.eligible_new_1l_patient_pool AS target_eligible_new_1l_patient_pool,

        target.competitor_overlap_pressure_truth AS target_competitor_overlap_pressure_truth,
        target.competitor_overlap_known_from_month AS target_competitor_overlap_known_from_month

    FROM origin_horizon_panel ohp

    INNER JOIN analytical_mart target
        ON ohp.therapy = target.therapy
       AND ohp.region = target.region
       AND ohp.target_month_index = target.month_index
),

point_in_time_features AS (
    SELECT
        therapy,
        region,
        forecast_origin_month,
        forecast_origin_month_index,
        horizon_months,
        target_month,
        target_month_index,

        target_observed_sales_units,
        target_latent_demand_units,

        target_market_regime,
        target_supply_constrained,

        sales_at_forecast_origin,
        sales_lag_1,
        sales_lag_3,
        sales_lag_6,
        sales_lag_12,
        rolling_mean_3,
        rolling_mean_6,
        rolling_sd_6,
        sales_change_vs_3_months_ago,
        therapy_age_months,
        calendar_month,
        calendar_year,

        CASE
            WHEN target_access_known_from_month <= forecast_origin_month_index
            THEN target_access_rate_truth
            ELSE NULL
        END AS target_access_rate_allowed,

        CASE
            WHEN target_access_known_from_month <= forecast_origin_month_index
            THEN 1
            ELSE 0
        END AS target_access_available_flag,

        CASE
            WHEN target_epidemiology_known_from_month <= forecast_origin_month_index
            THEN target_new_metastatic_diagnoses
            ELSE NULL
        END AS target_new_metastatic_diagnoses_allowed,

        CASE
            WHEN target_epidemiology_known_from_month <= forecast_origin_month_index
            THEN target_eligible_new_1l_patient_pool
            ELSE NULL
        END AS target_eligible_new_1l_patient_pool_allowed,

        CASE
            WHEN target_epidemiology_known_from_month <= forecast_origin_month_index
            THEN 1
            ELSE 0
        END AS target_epidemiology_available_flag,

        CASE
            WHEN target_competitor_overlap_known_from_month IS NULL THEN 0
            WHEN target_competitor_overlap_known_from_month <= forecast_origin_month_index
            THEN target_competitor_overlap_pressure_truth
            ELSE NULL
        END AS target_competitor_overlap_pressure_allowed,

        CASE
            WHEN target_competitor_overlap_known_from_month IS NULL THEN 1
            WHEN target_competitor_overlap_known_from_month <= forecast_origin_month_index
            THEN 1
            ELSE 0
        END AS target_competitor_overlap_available_flag,

        target_access_known_from_month,
        target_epidemiology_known_from_month,
        target_competitor_overlap_known_from_month

    FROM target_joined

    WHERE forecast_origin_month_index >= 13
)

SELECT *
FROM point_in_time_features
ORDER BY
    therapy,
    region,
    forecast_origin_month_index,
    horizon_months
"""

modeling_dataset = con.execute(modeling_dataset_sql).df()

print(f"Point-in-time modeling dataset created: {len(modeling_dataset):,} rows")
display(modeling_dataset.head())

con.register("modeling_dataset", modeling_dataset)

assert modeling_dataset["target_observed_sales_units"].notna().all()

assert (
    modeling_dataset["target_month_index"]
    == modeling_dataset["forecast_origin_month_index"]
    + modeling_dataset["horizon_months"]
).all()

assert modeling_dataset.groupby(
    ["therapy", "region", "forecast_origin_month_index", "horizon_months"]
).size().max() == 1

required_historical_features = [
    "sales_lag_1",
    "sales_lag_3",
    "sales_lag_6",
    "sales_lag_12",
    "rolling_mean_3",
    "rolling_mean_6",
]

assert modeling_dataset[required_historical_features].notna().all().all()

assert modeling_dataset.loc[
    modeling_dataset["target_access_available_flag"] == 0,
    "target_access_rate_allowed"
].isna().all()

assert modeling_dataset.loc[
    modeling_dataset["target_epidemiology_available_flag"] == 0,
    "target_new_metastatic_diagnoses_allowed"
].isna().all()

assert modeling_dataset.loc[
    modeling_dataset["target_competitor_overlap_available_flag"] == 0,
    "target_competitor_overlap_pressure_allowed"
].isna().all()

print("Point-in-time modeling dataset validation checks passed.")

In [ ]:
# ============================================================
# BLOCK 5 — Feature Provenance, Model Datasets, Save Outputs, Review Gate
# ============================================================

# Business purpose:
# Create final Notebook 2 outputs for Notebook 3.
#
# This block defines:
# - which features each model can use
# - feature provenance
# - feature availability audit
# - final CSV outputs
# - Notebook 2 senior review gate

historical_feature_columns = [
    "sales_at_forecast_origin",
    "sales_lag_1",
    "sales_lag_3",
    "sales_lag_6",
    "sales_lag_12",
    "rolling_mean_3",
    "rolling_mean_6",
    "rolling_sd_6",
    "sales_change_vs_3_months_ago",
    "therapy_age_months",
    "calendar_month",
    "calendar_year",
    "horizon_months",
]

patient_informed_feature_columns = [
    "target_access_rate_allowed",
    "target_new_metastatic_diagnoses_allowed",
    "target_eligible_new_1l_patient_pool_allowed",
    "target_competitor_overlap_pressure_allowed",
    "horizon_months",
    "calendar_month",
]

hybrid_feature_columns = historical_feature_columns + [
    "target_access_rate_allowed",
    "target_new_metastatic_diagnoses_allowed",
    "target_eligible_new_1l_patient_pool_allowed",
    "target_competitor_overlap_pressure_allowed",
]

model_feature_sets = {
    "historical_model": historical_feature_columns,
    "patient_informed_model": patient_informed_feature_columns,
    "hybrid_model": hybrid_feature_columns,
}

feature_provenance = pd.DataFrame([
    {
        "feature_name": "sales_at_forecast_origin",
        "feature_family": "historical_demand",
        "information_category": "observed_historical_information",
        "allowed_models": "historical_model, hybrid_model",
        "leakage_risk": "low if forecast origin is respected"
    },
    {
        "feature_name": "sales_lag_1 / sales_lag_3 / sales_lag_6 / sales_lag_12",
        "feature_family": "historical_demand",
        "information_category": "observed_historical_information",
        "allowed_models": "historical_model, hybrid_model",
        "leakage_risk": "low if lags are calculated before target month"
    },
    {
        "feature_name": "rolling_mean_3 / rolling_mean_6 / rolling_sd_6",
        "feature_family": "historical_demand",
        "information_category": "observed_historical_information",
        "allowed_models": "historical_model, hybrid_model",
        "leakage_risk": "medium if current or target month is accidentally included"
    },
    {
        "feature_name": "target_access_rate_allowed",
        "feature_family": "market_access",
        "information_category": "published_forward_information",
        "allowed_models": "patient_informed_model, hybrid_model",
        "leakage_risk": "high unless access_known_from_month <= forecast origin"
    },
    {
        "feature_name": "target_new_metastatic_diagnoses_allowed",
        "feature_family": "epidemiology",
        "information_category": "published_forward_information / planning assumption",
        "allowed_models": "patient_informed_model, hybrid_model",
        "leakage_risk": "high if actual future epidemiology is used before known"
    },
    {
        "feature_name": "target_eligible_new_1l_patient_pool_allowed",
        "feature_family": "patient_pool",
        "information_category": "published_forward_information / planning assumption",
        "allowed_models": "patient_informed_model, hybrid_model",
        "leakage_risk": "high if future patient pool truth is used before known"
    },
    {
        "feature_name": "target_competitor_overlap_pressure_allowed",
        "feature_family": "competition",
        "information_category": "published_forward_information",
        "allowed_models": "patient_informed_model, hybrid_model",
        "leakage_risk": "high unless competitor timing and access were known"
    },
    {
        "feature_name": "target_market_regime",
        "feature_family": "evaluation_label",
        "information_category": "evaluation_only",
        "allowed_models": "none",
        "leakage_risk": "must not be used as a model feature"
    },
])

feature_availability_audit = pd.DataFrame([
    {
        "feature": "target_access_rate_allowed",
        "availability_flag": "target_access_available_flag",
        "records_available": int(modeling_dataset["target_access_available_flag"].sum()),
        "records_not_available": int((modeling_dataset["target_access_available_flag"] == 0).sum()),
    },
    {
        "feature": "target_new_metastatic_diagnoses_allowed",
        "availability_flag": "target_epidemiology_available_flag",
        "records_available": int(modeling_dataset["target_epidemiology_available_flag"].sum()),
        "records_not_available": int((modeling_dataset["target_epidemiology_available_flag"] == 0).sum()),
    },
    {
        "feature": "target_competitor_overlap_pressure_allowed",
        "availability_flag": "target_competitor_overlap_available_flag",
        "records_available": int(modeling_dataset["target_competitor_overlap_available_flag"].sum()),
        "records_not_available": int((modeling_dataset["target_competitor_overlap_available_flag"] == 0).sum()),
    },
])

display(feature_provenance)
display(feature_availability_audit)

for model_name, features in model_feature_sets.items():
    assert "target_observed_sales_units" not in features
    assert "target_market_regime" not in features
    assert "target_supply_constrained" not in features

id_columns = [
    "therapy",
    "region",
    "forecast_origin_month",
    "forecast_origin_month_index",
    "horizon_months",
    "target_month",
    "target_month_index",
    "target_market_regime",
    "target_supply_constrained",
]

target_column = "target_observed_sales_units"

historical_modeling_dataset = modeling_dataset[
    id_columns + historical_feature_columns + [target_column]
].copy()

patient_informed_modeling_dataset = modeling_dataset[
    id_columns + patient_informed_feature_columns + [target_column]
].copy()

hybrid_modeling_dataset = modeling_dataset[
    id_columns + hybrid_feature_columns + [target_column]
].copy()

print("Model-ready datasets created:")
print(f"Historical dataset: {historical_modeling_dataset.shape}")
print(f"Patient-informed dataset: {patient_informed_modeling_dataset.shape}")
print(f"Hybrid dataset: {hybrid_modeling_dataset.shape}")

analytical_mart.to_csv("analytical_mart_therapy_region_month.csv", index=False)
modeling_dataset.to_csv("modeling_dataset.csv", index=False)
feature_provenance.to_csv("feature_provenance.csv", index=False)
feature_availability_audit.to_csv("feature_availability_audit.csv", index=False)

historical_modeling_dataset.to_csv("historical_modeling_dataset.csv", index=False)
patient_informed_modeling_dataset.to_csv("patient_informed_modeling_dataset.csv", index=False)
hybrid_modeling_dataset.to_csv("hybrid_modeling_dataset.csv", index=False)

print("Notebook 2 CSV outputs saved in the current Colab session.")

notebook_2_review_gate = pd.DataFrame([
    {
        "review_area": "Analytical grain",
        "assessment": "PASS",
        "comment": "The final modeling grain is therapy × region × forecast origin × horizon."
    },
    {
        "review_area": "SQL usage",
        "assessment": "PASS",
        "comment": "Uses CTEs, joins, CASE logic, aggregation, windows, lags, rolling calculations, and date parts."
    },
    {
        "review_area": "Forecast Information Set",
        "assessment": "PASS",
        "comment": "Features are separated into historical, published-forward, scenario-only, and forbidden information."
    },
    {
        "review_area": "Leakage prevention",
        "assessment": "PASS",
        "comment": "Future market and patient features are only populated when known from month is <= forecast origin."
    },
    {
        "review_area": "Model separation",
        "assessment": "PASS",
        "comment": "Historical, patient-informed, and hybrid feature sets are explicitly separated."
    },
    {
        "review_area": "Evaluation labels",
        "assessment": "PASS",
        "comment": "Market regime and supply constraint are kept for evaluation, not used as model features."
    },
    {
        "review_area": "Known limitation",
        "assessment": "LIMITATION",
        "comment": "Patient-informed features are planning-style synthetic projections, not real external data feeds."
    },
])

display(notebook_2_review_gate)
